## [Tutorial](https://github.com/biohub/esm/tree/main/cookbook/tutorials): How to run minibinder + scFv design fully end-to-end.

In this tutorial we will use [Modal](https://modal.com/) to parallelize binder design and synthesize a selection,
using the protocol described in the ESMC and ESMFold2 paper titled ["Language Modeling Materializes a World Model of Protein Biology"](https://biohub.ai/papers/esm_protein.pdf).

Biohub used this approach to design minibinders and scFvs against five therapeutically relevant targets — PDGFRB, EGFR, PD-L1, CD45, and CTLA4 — spanning receptor tyrosine kinases, immune checkpoints, and cell-surface phosphatases. Binders exhibit nanomolar affinity, target specificity, and functional activity in laboratory assays.

### One-time setup

In [ ]:
# Environment
! pip install esm@git+https://github.com/Biohub/esm.git@main
! pip install modal py3dmol pyarrow

In [ ]:
# Confirm you have a modal token, or make one
! modal token info  # Check
# ! modal token new  # Create

In [ ]:
# Deploy (or redeploy after changing modal_binder_design.py).
# This only needs to be run a single time, unless code in esmfold2_esmc_binder_design.py changes.
! modal deploy esmfold2_esmc_binder_design.py

### Imports

In [2]:
from itertools import product
from pathlib import Path

import modal
import pandas as pd
import py3Dmol
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from tqdm.auto import tqdm

### App setup

In [ ]:
ESMFold2Design = modal.Cls.from_name("esmfold2-design", "ESMFold2DesignModal")
# Set 'use_scaling_critics' to evaluate with the additional critics.
# Off by default. But cells below were populated with them enabled.
app = ESMFold2Design(use_scaling_critics=False)

### Run one job - interactive

In [4]:
# ---- Option 1: Use presets. ----
# Relies on the registry in modal_binder_design.py::{TARGET_SEQUENCES,BINDER_PROMPT_FACTORIES}, which can be modified.
future = app.design.spawn(target_name="ctla4", binder_name="minibinder")
future.get_dashboard_url()  # A clickable link to Modal dashboard

'https://modal.com/id/fc-01KSTCT9W9PYKN3HEKEZ168VJP'

In [5]:
# ---- Option 2: Provide your own sequences. ----
# Our pd-l1 sequence crop.
pdl1_sequence = "AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKVQHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNA"
# A sample of 'trastuzumab_framework_vhvl' template.  From esmfold2_esmc_binder_design.py::BINDER_PROMPT_FACTORIES.
trastuzumab_framework_vhvl = "EVQLVESGGGLVQPGGSLRLSCAAS#######YIHWVRQAPGKGLEWVARI#####TRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCSR###########WGQGTLVTVSSGGGSGGGSGGGSGGGSDIQMTQSPSSLSASVGDRVTITC###########WYQQKPGKAPKLLIY#######GVPSRFSGSRSGTDFTLTISSLQPEDFATYYC#########FGQGTKVEIK"
future2 = app.design.spawn(
    target_sequence=pdl1_sequence,
    binder_sequence=trastuzumab_framework_vhvl,
    is_antibody=True,
)
future2.get_dashboard_url()  # A clickable link to Modal dashboard

'https://modal.com/id/fc-01KSTCT9YCT8HH50718ZBABJRT'

In [ ]:
# ---- Monitor ----
# Tail a function's output here in jupyter
! modal app logs esmfold2-design -f --function-call {future2.object_id}

In [8]:
# ---- Load result ----
best_sequences, trajectory, critic_results = future2.get()
print("Best sequences: ", best_sequences)
df = pd.DataFrame(critic_results)
df.drop(columns=["logits", "complex"])

Best sequences:  ['AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKVQHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNA|EVQLVESGGGLVQPGGSLRLSCAASEPADEDDYIHWVRQAPGKGLEWVARITYEEKTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCSRWTAMAIGNDVAWGQGTLVTVSSGGGSGGGSGGGSGGGSDIQMTQSPSSLSASVGDRVTITCRFSQDVTIRLSWYQQKPGKAPKLLIYFAFILANGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCNYTRYSSSRFGQGTKVEIK']


,is_antibody,critic_name,batch_idx,designed_sequence,final_loss,iptm,distogram_iptm_proxy,cdr_distogram_iptm_proxy
0,True,ESMFold2-Experimental-Fast,0,AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...,4.544502,0.928495,0.850976,0.873329
1,True,ESMFold2-Experimental-Fast-Cutoff2025,0,AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...,4.544502,0.914886,0.837067,0.856937
2,True,ESMFold2-Experimental,0,AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...,4.544502,0.914534,0.824151,0.839054
3,True,ESMFold2-Experimental-Cutoff2025,0,AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...,4.544502,0.927602,0.835080,0.858924
4,True,ESMFold2-Experimental-Fast-base300M-step250k,0,AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...,4.544502,NaN,0.760568,0.791366
5,True,ESMFold2-Experimental-Fast-base300M-step500k,0,AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...,4.544502,NaN,0.817197,0.826138
6,True,ESMFold2-Experimental-Fast-base300M-step750k,0,AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...,4.544502,NaN,0.753613,0.773483
7,True,ESMFold2-Experimental-Fast-base300M-step1000k,0,AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...,4.544502,NaN,0.831106,0.860911
8,True,ESMFold2-Experimental-Fast-base300M-step1500k,0,AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...,4.544502,NaN,0.793353,0.737717
9,True,ESMFold2-Experimental-Fast-base600M-step250k,0,AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...,4.544502,NaN,0.794346,0.810242


In [9]:
# ---- Visualize ----
protein_complex = (
    df[df.critic_name.eq("ESMFold2-Experimental-Cutoff2025")].iloc[0].complex
)
(
    py3Dmol.view(width=600, height=600)
    .addModel(protein_complex.to_pdb_string(), "pdb")
    .setStyle({"chain": "A"}, {"cartoon": {"color": "green"}})  # pyright: ignore
    .setStyle({"chain": "B"}, {"cartoon": {"color": "cyan"}})  # pyright: ignore
    .addStyle(  # pyright: ignore
        {"not": {"atom": ["N", "CA", "C", "O"]}},
        {"stick": {"colorscheme": "default", "radius": 0.2}},
    )
    .center()  # pyright: ignore
    .zoomTo()  # pyright: ignore
)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### Run a sweep - async

In [6]:
# ---- Config ----
save_dir = Path("sweep")
save_dir.mkdir(exist_ok=True)

# Sweep settings - each key-value pair is an axis of a grid sweep.
line_sweeps = dict(
    target_name=["pd-l1"],
    target_sequence=[None],
    binder_name=["minibinder", "trastuzumab_framework_vhvl"],  # two modalities
    binder_sequence=[None],
    use_scaling_critics=[False],
    seed=list(range(8)),  # 8 seeds each
    batch_size=[1],
)
df = pd.DataFrame(product(*line_sweeps.values()), columns=list(line_sweeps.keys()))
display(df.head(2))
df.shape

,target_name,target_sequence,binder_name,binder_sequence,use_scaling_critics,seed,batch_size
0,pd-l1,None,minibinder,None,False,0,1
1,pd-l1,None,minibinder,None,False,1,1


(16, 7)

In [7]:
# ---- Launch ----
df["call_id"] = [
    app.design.spawn(
        target_name=row.target_name,
        target_sequence=row.target_sequence,
        binder_name=row.binder_name,
        binder_sequence=row.binder_sequence,
        seed=row.seed,
        batch_size=row.batch_size,
    ).object_id
    for row in df.itertuples()
]
df.to_parquet(save_dir / "manifest.parquet", index=False)
print(
    f"Spawned {len(df)} jobs. It is safe to close the notebook."
    "The next cell will resume from call_id's, saved by Modal for up to 7 days."
)

Spawned 16 jobs. It is safe to close the notebook.The next cell will resume from call_id's, saved by Modal for up to 7 days.


In [11]:
# ---- Monitor ----
df = pd.read_parquet(save_dir / "manifest.parquet")
df["future"] = df.call_id.transform(modal.FunctionCall.from_id)
df["status"] = df.future.transform(lambda f: f.get_call_graph()[0].status.name)
print("First task url: ", df.at[0, "future"].get_dashboard_url())  # pyright: ignore
df.status.value_counts()

First task url:  https://modal.com/id/fc-01KSTCTA27QYFGWNB67BPKZ72Z


status
SUCCESS    16
Name: count, dtype: int64

In [12]:
# ---- Collect ----
df["result"] = modal.FunctionCall.gather(
    *df.future.tolist()
)  # Blocks until all jobs are complete.
df["result_df"] = [pd.DataFrame(r[2]) for r in df.result]  # pyright: ignore

In [13]:
# ---- Select ----

# Join all result_df's, with other fields in df broadcasted as metadata.
df_result = pd.concat(
    [
        row.result_df.assign(**row.drop(["result", "result_df"]).to_dict())  # pyright: ignore
        for _, row in df.iterrows()
    ],
    ignore_index=True,
    axis=0,
)

# Filter minibinder designs with isoelectric point >= 6.
df_result["binder_sequence"] = df_result.designed_sequence.str.split(r"\|").str[1]
df_result["isoelectric_point"] = [
    ProteinAnalysis(seq).isoelectric_point()
    for seq in tqdm(df_result.binder_sequence.values)
]
# Isoelectric point filter
df_filter = df_result[df_result.is_antibody | df_result.isoelectric_point.lt(6)]


# Select the top 84 designs from each (target, binder) combination
def select(df: pd.DataFrame) -> pd.DataFrame:
    # Where the cdr-specific iptm proxy exists, use it (antibodies).
    # Else use the full distogram iptm proxy.
    # If neither exists (use_scaling_checkpoints=False), then there is no contribution from this term.
    df["iptm_proxy"] = df.cdr_distogram_iptm_proxy.combine_first(
        df.distogram_iptm_proxy
    ).fillna(0)
    df = df.groupby("designed_sequence", as_index=False).agg(
        dict(iptm="mean", iptm_proxy="mean")
    )
    df["selection_score"] = 0.5 * df.iptm + 0.5 * df.iptm_proxy
    return df.nlargest(min(len(df), 84), "selection_score")


df_select = df_filter.groupby(["target_name", "binder_name"]).apply(
    select, include_groups=False
)
df_select.to_parquet(save_dir / "selection.parquet", index=False)

  0%|          | 0/304 [00:00<?, ?it/s]

In [17]:
df_result[df_result.critic_name.eq("ESMFold2-Experimental-Cutoff2025")].drop(
    columns=["complex", "logits"]
)

,is_antibody,critic_name,batch_idx,designed_sequence,final_loss,iptm,distogram_iptm_proxy,cdr_distogram_iptm_proxy,target_name,target_sequence,binder_name,binder_sequence,use_scaling_critics,seed,batch_size,call_id,future,status,isoelectric_point
3,False,ESMFold2-Experimental-Cutoff2025,0,AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...,2.887006,0.949471,0.902141,NaN,pd-l1,None,minibinder,QSSDDEIDKEVNKVAAEIALAVAELTRAAADGDDKEVDKQLKKALK...,False,0,1,fc-01KSTCTA27QYFGWNB67BPKZ72Z,FunctionCall.from_id('fc-01KSTCTA27QYFGWNB67BP...,SUCCESS,9.521739
22,False,ESMFold2-Experimental-Cutoff2025,0,AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...,4.945910,0.392557,0.610550,NaN,pd-l1,None,minibinder,KWEIWRLLWKIGNNLWNNNNNNNNWNAIWTIWWWLIWWLIWWLLIN...,False,1,1,fc-01KSTCTA52ZQZXC2RZ4F12ZJNC,FunctionCall.from_id('fc-01KSTCTA52ZQZXC2RZ4F1...,SUCCESS,10.605259
41,False,ESMFold2-Experimental-Cutoff2025,0,AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...,4.628414,0.920426,0.857930,NaN,pd-l1,None,minibinder,SIIRILIIIVIKAIKKVSKIAKILKKALKELAKSGASKEIVEILIE...,False,2,1,fc-01KSTCTA7ZBCYQZCJ1DHGT94BZ,FunctionCall.from_id('fc-01KSTCTA7ZBCYQZCJ1DHG...,SUCCESS,10.170871
60,False,ESMFold2-Experimental-Cutoff2025,0,AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...,4.770513,0.903192,0.831106,NaN,pd-l1,None,minibinder,MSLEELLKEIVEALKSGDFKKAAKAIKEAAKIIFSENIEVASAKIL...,False,3,1,fc-01KSTCTAASYNGFY59G85TYT5SZ,FunctionCall.from_id('fc-01KSTCTAASYNGFY59G85T...,SUCCESS,7.856585
79,False,ESMFold2-Experimental-Cutoff2025,0,AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...,3.711260,0.914334,0.849982,NaN,pd-l1,None,minibinder,QNSNNNNNNNNEEDEEIDIKILKILIKLLIIIILLKKSPSSSSKKK...,False,4,1,fc-01KSTCTAEAAQN1QKMYCDD92HKN,FunctionCall.from_id('fc-01KSTCTAEAAQN1QKMYCDD...,SUCCESS,9.874187
98,False,ESMFold2-Experimental-Cutoff2025,0,AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...,3.959803,0.934015,0.855943,NaN,pd-l1,None,minibinder,SLILNILNIRINEINNLITNASKNELILYLKNLNIILKILLILLQN...,False,5,1,fc-01KSTCTAHDBY4M10ESJP0HVS5Z,FunctionCall.from_id('fc-01KSTCTAHDBY4M10ESJP0...,SUCCESS,5.117010
117,False,ESMFold2-Experimental-Cutoff2025,0,AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...,4.276257,0.936331,0.892703,NaN,pd-l1,None,minibinder,LLELLKILVKNAKNFSSSELYIVIMLLEILSNEDPREALILVEEII...,False,6,1,fc-01KSTCTAM1V79QCSF9612BD1RV,FunctionCall.from_id('fc-01KSTCTAM1V79QCSF9612...,SUCCESS,4.560045
136,False,ESMFold2-Experimental-Cutoff2025,0,AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...,3.663691,0.846763,0.783418,NaN,pd-l1,None,minibinder,QQLQLLIIQLILLIIVKILLQIANILLQEAKLSDSDDSEKIIKTLK...,False,7,1,fc-01KSTCTAP3G7YRJ1RSQ2NFSTB5,FunctionCall.from_id('fc-01KSTCTAP3G7YRJ1RSQ2N...,SUCCESS,9.399378
155,True,ESMFold2-Experimental-Cutoff2025,0,AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...,3.823050,0.928952,0.839054,0.851969,pd-l1,None,trastuzumab_framework_vhvl,EVQLVESGGGLVQPGGSLRLSCAASSDRSYSVSYIHWVRQAPGKGL...,False,0,1,fc-01KSTCTAR7DKSNJXBHARD4FFZ2,FunctionCall.from_id('fc-01KSTCTAR7DKSNJXBHARD...,SUCCESS,6.984682
174,True,ESMFold2-Experimental-Cutoff2025,0,AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...,4.117104,0.908898,0.781431,0.797327,pd-l1,None,trastuzumab_framework_vhvl,EVQLVESGGGLVQPGGSLRLSCAASEPLSYRIYIHWVRQAPGKGLE...,False,1,1,fc-01KSTCTATESMP6KMYPFTQ1B2PF,FunctionCall.from_id('fc-01KSTCTATESMP6KMYPFTQ...,SUCCESS,8.632139


In [15]:
df_select

designed_sequence  \
target_name binder_name                                                                       
pd-l1       minibinder                 0  AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...   
                                       1  AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...   
            trastuzumab_framework_vhvl 4  AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...   
                                       3  AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...   
                                       6  AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...   
                                       5  AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...   
                                       1  AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...   
                                       0  AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...   
                                       2  AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...   
                                       7  AFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKN...   

                                              iptm  iptm_proxy  \
target_name binder_name                                          
pd-l1       minibinder                 0  0.930100    0.856649   
                                       1  0.937393    0.837694   
            trastuzumab_framework_vhvl 4  0.925267    0.810059   
                                       3  0.919929    0.788386   
                                       6  0.917205    0.780960   
                                       5  0.906416    0.753404   
                                       1  0.904410    0.719259   
                                       0  0.877860    0.723024   
                                       2  0.868486    0.710997   
                                       7  0.763347    0.644642   

                                          selection_score  
target_name binder_name                                    
pd-l1       minibinder                 0         0.893375  
                                       1         0.887544  
            trastuzumab_framework_vhvl 4         0.867663  
                                       3         0.854157  
                                       6         0.849083  
                                       5         0.829910  
                                       1         0.811835  
                                       0         0.800442  
                                       2         0.789742  
                                       7         0.703995

## Appendix

### Modal Primer

- **info: ephemeral vs deployment**  
  Ephemeral = temporary app from `modal run` or `app.run()`, stopped when the client exits. Deployment = persistent named app from `modal deploy`, reused and observable across runs. ([modal.com](https://modal.com/docs/guide/apps?utm_source=openai))

- **info: dashboard**  
  Generic dashboard/apps page: [https://modal.com/apps](https://modal.com/apps). Modal also prints app/deployment links during runs/deploys. ([modal.com](https://modal.com/docs/guide/apps?utm_source=openai))

- **cli: ephemeral run**  
  ```bash
  modal run path/to/app.py
  ```

- **cli: deploy/redeploy**  
  ```bash
  modal deploy path/to/app.py
  ```
  Running this on an existing app name redeploys a new version. ([modal.com](https://modal.com/docs/reference/cli/deploy?utm_source=openai))

- **local: ephemeral from Python**  
  ```python
  with modal.enable_output():
      with modal_app.run():
          result = local_modal_obj.remote(...)
  ```

- **local: call a deployment**  
  ```python
  Cls = modal.Cls.from_name("app-name", "ClassName")
  obj = Cls(...)
  result = obj.method.remote(...)
  ```
  `Cls.from_name` references a class from a deployed app lazily. ([modal.com](https://modal.com/docs/reference/modal.Cls?utm_source=openai))